In [1]:
# load data and build index 

from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [3]:
# load Gold Standard data

import pandas as pd

df_ground_truth = pd.read_csv("data/ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [4]:
def search_boost(query, question_boost):
    boost_dict = {"question": question_boost, "section": 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )


q = ground_truth[0]


doc_id = q["document"]
results = search_boost(query=q["question"], question_boost=5.0)
print(q["question"])
print(results)

Is it too late to sign up for the classes?
[{'id': '0fab61eca2', 'course': 'llm-zoomcamp', 'section': 'Capstone Project', 'question': 'Is it a group project?', 'answer': 'No, the capstone is an individual project.\n\nYou can collaborate or discuss a larger idea with other students, but each submitted project must stand on its own. A shared system can work only if it is clearly decomposed into independent projects, where each person has a separate qualifying component and a separate repository.\n\nIf the work cannot be evaluated independently for each participant, it does not satisfy the project requirement.'}, {'id': '6bceeebf04', 'course': 'llm-zoomcamp', 'section': 'Workshop: Open-Source Data Ingestion (dlt)', 'question': 'How to set up a new dlt project when loading from cloud?', 'answer': 'Start with the following command on the command line:\n\n```bash\n dlt init filesystem duckdb\n```\n\nMore directions can be found at [dlthub.com](https://dlthub.com/docs/tutorial/filesystem)'}, 

In [ ]:
for boost in [0.5, 1.0, 3.0, 5.0, 10.0]:
    result = evaluate(
        ground_truth,
        lambda query, boost=boost: search_boost(query, boost)
    )
    print(f"boost={boost}: {result}")